In [40]:
import torch
from torch import nn
import os
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

In [28]:
import splitfolders
splitfolders.ratio(input="data",output="dataset_split",seed=42,ratio=(.8,.0,.2))

Copying files: 49779 files [00:03, 15076.57 files/s]


In [54]:
from pathlib import Path
data_path = Path("")
image_path = data_path / "dataset_split"
train_path = image_path / "train"
test_path = image_path / "test"

In [34]:
def check_data(dir_path):
    for dirpath,dirnames,filenames in os.walk(dir_path):
        print(f"# of directories in '{len(dirnames)}' and {len(filenames)} images in {dirpath}")



In [36]:
check_data(image_path)

# of directories in '2' and 0 images in dataset_split
# of directories in '7' and 0 images in dataset_split/test
# of directories in '0' and 1184 images in dataset_split/test/surprise
# of directories in '0' and 1184 images in dataset_split/test/disgust
# of directories in '0' and 1184 images in dataset_split/test/angry
# of directories in '0' and 1307 images in dataset_split/test/sad
# of directories in '0' and 1184 images in dataset_split/test/fear
# of directories in '0' and 1634 images in dataset_split/test/neutral
# of directories in '0' and 2280 images in dataset_split/test/happy
# of directories in '7' and 0 images in dataset_split/train
# of directories in '0' and 4736 images in dataset_split/train/surprise
# of directories in '0' and 4736 images in dataset_split/train/disgust
# of directories in '0' and 4736 images in dataset_split/train/angry
# of directories in '0' and 5228 images in dataset_split/train/sad
# of directories in '0' and 4736 images in dataset_split/train/fear


In [55]:
NUM_WORKERS = os.cpu_count()

def create_dataloader(train_dir,
                      test_dir,
                      transforms: transforms.Compose,
                      batch_size:int,
                      workers:int = NUM_WORKERS):
    
    train_data = datasets.ImageFolder(root=train_dir,
                                      transform=transforms)
    test_data = datasets.ImageFolder(root=test_dir,
                                     transform=transforms)
    
    class_names = train_data.classes

    train_dataloader = DataLoader(dataset=train_data,
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=workers)
    
    test_dataloader = DataLoader(dataset=test_data,
                                  batch_size=batch_size,
                                  shuffle=True,
                                  num_workers=workers)
    return train_dataloader, test_dataloader, class_names

In [56]:
weight = models.EfficientNet_V2_S_Weights.DEFAULT

In [57]:
auto_transforms = weight.transforms()

In [58]:
auto_transforms.crop_size = [96]
auto_transforms.resize_size = [96]

In [59]:
auto_transforms

ImageClassification(
    crop_size=[96]
    resize_size=[96]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [60]:
train_dataloader, test_dataloader, class_names = create_dataloader(train_dir=train_path,
                                                                   test_dir=test_path,
                                                                   transforms=auto_transforms,
                                                                   batch_size=32,)

In [66]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [77]:
model = models.efficientnet_v2_s(weights=weight).to(device)

In [78]:
from torchinfo import summary
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 1000]                --                        True
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 1280, 3, 3]          --                        True
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 24, 48, 48]          --                        True
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 24, 48, 48]          648                       True
│    │    └─BatchNorm2d: 3-2                            [32, 24, 48, 48]          [32, 24, 48, 48]          48                        True
│    │    └─SiLU: 3-3                                   [32, 24, 48, 48]          [32, 24, 48, 48]          --                        --
│    └─Sequential: 2-2  

In [79]:
for params in model.parameters():
    params.requires_grad = False

In [80]:
summary(model=model,input_size=(32,3,96,96),col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
EfficientNet                                            [32, 3, 96, 96]           [32, 1000]                --                        False
├─Sequential: 1-1                                       [32, 3, 96, 96]           [32, 1280, 3, 3]          --                        False
│    └─Conv2dNormActivation: 2-1                        [32, 3, 96, 96]           [32, 24, 48, 48]          --                        False
│    │    └─Conv2d: 3-1                                 [32, 3, 96, 96]           [32, 24, 48, 48]          (648)                     False
│    │    └─BatchNorm2d: 3-2                            [32, 24, 48, 48]          [32, 24, 48, 48]          (48)                      False
│    │    └─SiLU: 3-3                                   [32, 24, 48, 48]          [32, 24, 48, 48]          --                        --
│    └─Sequential: 